# Huấn luyện và Lưu trữ Mô hình Tốt nhất (Best Models)
—
Notebook này dựa trên bản chạy thử của `advanced_models.py` để làm sạch dữ liệu, chuẩn bị features (kể cả Bayesian Target Encoding) và huấn luyện trên thuật toán tốt nhất là **CatBoost**. Cuối cùng, nó kết xuất Pipeline để phần web Dashboard App có thể tái sử dụng để dự báo.

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor
import joblib
import json
import os

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
# =====================================================================
# 1. ĐỌC VÀ LÀM SẠCH DỮ LIỆU
# =====================================================================
print("=" * 60)
print("📊 BƯỚC 1: ĐỌC VÀ LÀM SẠCH DỮ LIỆU")
print("=" * 60)

df = pd.read_csv('../data/filtered_data.csv')
df = df.drop_duplicates(keep='first')
df['price_billion'] = df['price_total'] / 1e9

# Chuẩn hóa chuỗi
categorical_cols = ['category', 'province', 'district', 'legal_status']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('nan', 'Unknown')
        df[col] = df[col].fillna('Unknown')

print(f"  Dữ liệu gốc: {df.shape[0]:,} dòng x {df.shape[1]} cột")

# =====================================================================
# 2. LỌC NGOẠI LAI (OUTLIER REMOVAL)
# =====================================================================
nha_categories = ['Nhà riêng', 'Căn hộ chung cư', 'Nhà mặt phố', 'Nhà biệt thự, liền kề']
dat_categories = ['Bán đất', 'Đất nền dự án']

cond_nha = (df['category'].isin(nha_categories)) & (df['num_bedrooms'] > 0) & (df['num_toilets'] > 0)
cond_dat = df['category'].isin(dat_categories)
df = df[cond_nha | cond_dat]

cond_filter = (
    (df['area'] >= 15) & (df['area'] <= 1000) &
    (df['price_billion'] >= 0.3) & (df['price_billion'] <= 80)
)
df = df[cond_filter]

df_parts = []
for cat in df['category'].unique():
    sub = df[df['category'] == cat].copy()
    Q1 = sub['price_billion'].quantile(0.02)
    Q3 = sub['price_billion'].quantile(0.98)
    sub = sub[(sub['price_billion'] >= Q1) & (sub['price_billion'] <= Q3)]
    df_parts.append(sub)
df_clean = pd.concat(df_parts, ignore_index=True)

print(f"  Sau lọc ngoại lai: {df_clean.shape[0]:,} dòng")

# --> LƯU LẠI DATA ĐÃ CLEAN CHO BẢN ĐỒ DASHBOARD EDA
df_clean.to_csv('../data/dashboard_data.csv', index=False)
print("  Đã lưu dữ liệu chuẩn để vẽ biểu đồ và EDA vào data/dashboard_data.csv")


📊 BƯỚC 1: ĐỌC VÀ LÀM SẠCH DỮ LIỆU
  Dữ liệu gốc: 32,158 dòng x 17 cột
  Sau lọc ngoại lai: 24,548 dòng
  Đã lưu dữ liệu chuẩn để vẽ biểu đồ và EDA vào data/dashboard_data.csv


In [3]:
# =====================================================================
# 3. FEATURE ENGINEERING (TẠO ĐẶC TRƯNG)
# =====================================================================
print("\n" + "=" * 60)
print("🔧 BƯỚC 3: FEATURE ENGINEERING")
print("=" * 60)

df_clean['total_rooms'] = df_clean['num_bedrooms'] + df_clean['num_toilets']
df_clean['area_per_room'] = df_clean['area'] / (df_clean['total_rooms'] + 1)
df_clean['area_per_floor'] = df_clean['area'] / (df_clean['num_floors'] + 1)
df_clean['area_per_bed'] = df_clean['area'] / (df_clean['num_bedrooms'] + 1)
df_clean['toilet_bed_ratio'] = df_clean['num_toilets'] / (df_clean['num_bedrooms'] + 1)
df_clean['amenity_score'] = df_clean['num_schools_1km'] + df_clean['num_hospitals_2km'] + df_clean['num_markets_1km']
df_clean['log_area'] = np.log1p(df_clean['area'])
df_clean['area_x_frontage'] = df_clean['area'] * df_clean['frontage'].fillna(0)
df_clean['frontage_ratio'] = df_clean['frontage'].fillna(0) / (df_clean['area'] + 1)
df_clean['rooms_per_floor'] = df_clean['total_rooms'] / (df_clean['num_floors'] + 1)

print(f"  Tổng số đặc trưng: {df_clean.shape[1]}")


🔧 BƯỚC 3: FEATURE ENGINEERING
  Tổng số đặc trưng: 27


In [4]:
# =====================================================================
# 4. CHUẨN BỊ DỮ LIỆU & TARGET MỤC TIÊU
# =====================================================================
cols_to_drop = ['price_total', 'price_billion']
X = df_clean.drop(columns=cols_to_drop)
y = df_clean['price_billion']

y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
y_train = np.expm1(y_train_log)
y_test = np.expm1(y_test_log)


In [5]:
# =====================================================================
# 5. BAYESIAN TARGET ENCODING
# =====================================================================
train_temp = X_train.copy()
train_temp['price_per_m2'] = y_train / train_temp['area']

global_median_price_m2 = train_temp['price_per_m2'].median()
SMOOTHING_WEIGHT = 20

dist_cat_stats = train_temp.groupby(['district', 'category'])['price_per_m2'].agg(['median', 'count'])
dist_cat_stats['smoothed'] = ((dist_cat_stats['count'] * dist_cat_stats['median']) + (SMOOTHING_WEIGHT * global_median_price_m2)) / (dist_cat_stats['count'] + SMOOTHING_WEIGHT)
dist_cat_dict = dist_cat_stats['smoothed'].to_dict()

dist_stats = train_temp.groupby('district')['price_per_m2'].agg(['median', 'count'])
dist_stats['smoothed'] = ((dist_stats['count'] * dist_stats['median']) + (SMOOTHING_WEIGHT * global_median_price_m2)) / (dist_stats['count'] + SMOOTHING_WEIGHT)
dist_dict = dist_stats['smoothed'].to_dict()

prov_stats = train_temp.groupby('province')['price_per_m2'].agg(['median', 'count'])
prov_stats['smoothed'] = ((prov_stats['count'] * prov_stats['median']) + (SMOOTHING_WEIGHT * global_median_price_m2)) / (prov_stats['count'] + SMOOTHING_WEIGHT)
prov_dict = prov_stats['smoothed'].to_dict()

def apply_target_encoding(df_input):
    df_out = df_input.copy()
    tuples = list(zip(df_out['district'].astype(str), df_out['category'].astype(str)))
    enc1 = pd.Series([str(t) for t in tuples]).map({str(k): v for k,v in dist_cat_dict.items()})
    enc2 = df_out['district'].astype(str).map(dist_dict)
    enc3 = df_out['province'].astype(str).map(prov_dict)
    df_out['expected_price_m2'] = enc1.fillna(enc2).fillna(enc3).fillna(global_median_price_m2)
    df_out['expected_price'] = df_out['area'] * df_out['expected_price_m2']
    df_out['log_expected_price'] = np.log1p(df_out['expected_price'])
    df_out.drop(columns=['expected_price_m2'], inplace=True)
    return df_out

X_train = apply_target_encoding(X_train)
X_test = apply_target_encoding(X_test)

# Lưu TỪ ĐIỂN TARGET ENCODING LẠI ĐỂ APP STREAMLIT CÓ THỂ TÁI SỬ DỤNG
dictionaries = {
    'global_median_price_m2': global_median_price_m2,
    'dist_cat_dict': {str(k): v for k, v in dist_cat_dict.items()},
    'dist_dict': dist_dict,
    'prov_dict': prov_dict
}
with open('../data/encoding_dicts.json', 'w', encoding='utf-8') as f:
    json.dump(dictionaries, f, ensure_ascii=False)
print("  Đã xuất metadata Target Encoding vào data/encoding_dicts.json")


  Đã xuất metadata Target Encoding vào data/encoding_dicts.json


In [6]:
# =====================================================================
# 6. TRAIN MÔ HÌNH VỚI CATBOOST
# =====================================================================
print("\n" + "=" * 60)
print("🚀 BƯỚC 6: TRAIN VÀ XUẤT MÔ HÌNH")
print("=" * 60)

cat_features = [col for col in categorical_cols if col in X_train.columns]
for col in cat_features:
    X_train[col] = X_train[col].astype(str).fillna('Unknown')
    X_test[col] = X_test[col].astype(str).fillna('Unknown')

cat_indices = [X_train.columns.get_loc(col) for col in cat_features]

model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=150
)

model.fit(
    X_train, y_train_log,
    cat_features=cat_indices,
    eval_set=(X_test, y_test_log),
    use_best_model=True
)

preds_log = model.predict(X_test)
preds = np.expm1(preds_log)

r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print("\n🏆 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (Test Data):")
print(f"  R²   : {r2:.4f}")
print(f"  RMSE : {rmse:.4f} Tỷ VNĐ")
print(f"  MAE  : {mae:.4f} Tỷ VNĐ")

joblib.dump(model, '../data/best_catboost_model.joblib')
print("\n✅ Đã xuất mô hình thành công tại `data/best_catboost_model.joblib`")


🚀 BƯỚC 6: TRAIN VÀ XUẤT MÔ HÌNH
0:	learn: 0.0367086	test: 0.0356128	best: 0.0356128 (0)	total: 253ms	remaining: 12m 38s
200:	learn: 0.8546491	test: 0.8382437	best: 0.8382437 (200)	total: 14.6s	remaining: 3m 23s
400:	learn: 0.8808849	test: 0.8592646	best: 0.8592646 (400)	total: 28.1s	remaining: 3m 1s
600:	learn: 0.8961768	test: 0.8690826	best: 0.8690826 (600)	total: 44.6s	remaining: 2m 58s
800:	learn: 0.9072478	test: 0.8750616	best: 0.8750616 (800)	total: 59.3s	remaining: 2m 42s
1000:	learn: 0.9155097	test: 0.8789408	best: 0.8789408 (1000)	total: 1m 27s	remaining: 2m 54s
1200:	learn: 0.9223428	test: 0.8816027	best: 0.8816027 (1200)	total: 1m 42s	remaining: 2m 32s
1400:	learn: 0.9277833	test: 0.8835526	best: 0.8835526 (1400)	total: 1m 55s	remaining: 2m 12s
1600:	learn: 0.9323609	test: 0.8851348	best: 0.8851348 (1600)	total: 2m 10s	remaining: 1m 54s
1800:	learn: 0.9365347	test: 0.8862866	best: 0.8862875 (1799)	total: 2m 24s	remaining: 1m 36s
2000:	learn: 0.9401512	test: 0.8873522	best: 0